# Trees, Forests and Model Comparison
### BAI709 Data Modelling and Machine Learning

---

**Two business questions run through this notebook.**

1. *A lender* already has a logistic regression that ranks home equity applicants by risk. Is a more complex model worth having, and would you trust one that nobody can read?
2. *A telecoms retention team* wants to know whether a forest of trees beats the simple model at predicting which customers will leave. Does it matter which model they choose?

You will build, test and compare three kinds of model on the same data with the same split, so any difference comes from the model and not from luck.

---

### How this notebook works

Every part follows the same five steps, so you always know where you are.

| Step | What it is |
|---|---|
| 1 | **Load and look**: see the data before you model it |
| 2 | **Run the model**: one short block of code |
| 3 | **Read the output in plain English**: what the numbers say, and what to look for first |
| 4 | **Check against JMP**: the same result from a second tool |
| 5 | **Say it to a manager**: two sentences in business language |

- Run cells in order with **Shift+Enter**. The data files are in `../data/`.
- **Copilot is on.** Where a cell says *Ask Copilot*, write the prompt, read the code it gives you, run it, and check the answer against the result shown in this notebook or in JMP. Code you have not checked is a guess.
- Cells marked **Your turn** are for you to change and re-run. Cells marked **Stretch** are blank on purpose, for quick finishers who want to build something of their own.
- All money figures in this notebook (loan losses, retention offers) are **invented for teaching**. Real costs would come from the business.

## Setup

One cell for the libraries, and one cell for every fixed setting. The fixed settings are the same in every notebook in this module, so differences between models come from the models.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("Libraries loaded.")

### The fixed settings (read this cell once, then leave it alone)

- **Split:** 70% training, 30% test, `random_state=1`, stratified so both parts keep the same share of defaults. The split happens **before** any cleaning.
- **Logistic regression:** inputs standardised, then `LogisticRegression(max_iter=5000)`.
- **Tree, depth 5:** `max_depth=5, min_samples_leaf=50`. **Unconstrained tree:** no limits at all.
- **Forest:** 300 trees, `min_samples_leaf=5`.
- **Cross-validation:** five folds on the training rows only, scored by AUC.
- **Costs (invented):** a missed default loses 10,000; turning away a good applicant loses 1,000. The best cut-off from those two numbers is 1,000 / (10,000 + 1,000) = 0.091.

> **A note on the baseline.** This notebook uses scikit-learn's logistic regression, which applies a mild penalty by default. The model you fitted with statsmodels in the logistic regression notebook applies none, so a few cells here differ slightly from that notebook (for example one applicant moves between two boxes of the confusion matrix). The conclusions do not change.

In [ ]:
SPLIT = dict(test_size=0.3, random_state=1)            # stratify=y is added where we split
CV = StratifiedKFold(5, shuffle=True, random_state=1)  # used on the training rows only

def make_models():
    # A fresh, unfitted set of the four models every time we call it
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        "Tree, depth 5": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=1),
        "Tree, unconstrained": DecisionTreeClassifier(random_state=1),
        "Random forest, 300 trees": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                           random_state=1, n_jobs=-1),
    }

COST_MISSED_DEFAULT = 10_000    # invented: approving a loan that defaults
COST_DECLINED_GOOD = 1_000      # invented: turning away a good applicant
LENDER_CUTOFF = round(COST_DECLINED_GOOD / (COST_MISSED_DEFAULT + COST_DECLINED_GOOD), 3)   # 0.091
print(f"The lender's cut-off from the two costs: {LENDER_CUTOFF:.3f}")

---
# Demo: will this applicant default? (loan applications)

## Part 1: Look at the data and prepare it

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Count applicants, defaults and gaps. Spot that the gaps matter. |
| 2 | **Run the model** | Split first, then clean using the training rows only. |
| 3 | **Read the output in plain English** | Check the shapes and the medians in a sentence. |
| 4 | **Check against JMP** | Export the prepared table with a `Split` and a `Validation` column so JMP uses the same split. |
| 5 | **Say it to a manager** | Say why a missing value can be a clue. |

### Step 1: Load and look

**What to look for:** the share of applicants who default, and which columns have gaps. A gap is not always an accident.

In [ ]:
hmeq = pd.read_csv("../data/hmeq_full.csv")
print(f"{len(hmeq):,} applicants and {hmeq.shape[1]} columns")
print(f"Default rate (BAD = 1): {hmeq['BAD'].mean():.1%}")
hmeq.head()

In [ ]:
# How many gaps in each column?
gaps = hmeq.isna().sum()
gaps[gaps > 0].sort_values(ascending=False)

In [ ]:
# Do the applicants with no debt-to-income figure behave differently?
missing_dti = hmeq["DEBTINC"].isna()
rate_missing = hmeq.loc[missing_dti, "BAD"].mean()
rate_present = hmeq.loc[~missing_dti, "BAD"].mean()
print(f"Default rate where DEBTINC is present: {rate_present:.1%}")
print(f"Default rate where DEBTINC is missing: {rate_missing:.1%}")
print(f"{missing_dti.sum():,} applicants have no debt-to-income figure.")

**What it says:** the applicants with no debt-to-income figure default far more often than the rest. The gap is information, so we will keep a flag for it instead of throwing those rows away.

> **Concept: train and test.** We split the applicants into a **training set** (the model learns from it) and a **test set** (held back, used once, to see how the model does on applicants it has never met). If we judged a model on the rows it learned from, we would be measuring memory, not skill.
> `stratify=y` keeps the same share of defaults in both parts.
>
> **Order matters.** Split first. Anything the preparation *learns* from the data (a median, a frequency, a scaling) must be learned from the training rows only and then applied to the test rows. If the test rows help set the median, test information has leaked into training, and the test is no longer a fair test. That is called **leakage**.

> **Concept: one-hot (indicator) encoding.** Models work on numbers, so a column such as `JOB` becomes one 0/1 column per category. We drop one category per column (the *reference level*) because it is implied when all the others are 0. Trees do not need a reference level, but we keep the same columns in every model so that the comparison is like for like.

> **Concept: frequency encoding.** A column with hundreds of categories (an agent ID, a postcode) would make hundreds of indicator columns. Frequency encoding replaces each category with how common it is. The frequencies must be counted **on the training rows only**, as in the cell below. Counting them on the whole file before the split is a quiet form of leakage. We show it on `JOB` purely to see the idea; our models use indicator columns.

In [ ]:
# Frequency encoding, done properly: learn on training rows, apply to test rows.
tmp_train, tmp_test = train_test_split(hmeq, stratify=hmeq["BAD"], **SPLIT)
job_freq = tmp_train["JOB"].value_counts(normalize=True)          # learned from training rows only
demo = pd.DataFrame({"JOB": tmp_test["JOB"].head(6),
                     "JOB as frequency": tmp_test["JOB"].head(6).map(job_freq).fillna(0).round(3)})
demo

### Step 2: Run the preparation

This is the preparation from the logistic regression notebook, in one cell. It is a function so every later part can use it again.

1. Split first, stratified, with the fixed settings.
2. Add `DEBTINC_missing` (1 where the debt-to-income figure is blank) **before** filling anything.
3. Fill the other blanks in numeric columns with the **training** median, and give the test rows the same training medians.
4. Blank `REASON` and `JOB` become a "Missing" category; indicator coding with reference levels `DebtCon` and `Other`.
5. Leave the money columns (`LOAN`, `MORTDUE`, `VALUE`) in the units recorded in the file. Trees do not care about the size of a number, and the logistic pipeline standardises its inputs, so rescaling to thousands would change nothing that matters.

In [ ]:
def prepare_hmeq(data):
    y = data["BAD"]
    X = data.drop(columns="BAD")
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, **SPLIT)    # split FIRST

    numeric = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]
    medians = X_train[numeric].median()                                              # learned from training only

    cleaned = []
    for part in (X_train, X_test):
        part = part.copy()
        part["DEBTINC_missing"] = part["DEBTINC"].isna().astype(int)                 # flag BEFORE filling
        part[numeric] = part[numeric].fillna(medians)
        for col in ("REASON", "JOB"):
            part[col] = part[col].fillna("Missing")
        cleaned.append(part)
    X_train, X_test = cleaned

    X_train = pd.get_dummies(X_train, columns=["REASON", "JOB"], dtype=int)
    X_test = pd.get_dummies(X_test, columns=["REASON", "JOB"], dtype=int).reindex(columns=X_train.columns, fill_value=0)
    drop = ["REASON_DebtCon", "JOB_Other"]                                           # reference levels
    return X_train.drop(columns=drop), X_test.drop(columns=drop), y_train, y_test, medians

X_train, X_test, y_train, y_test, medians = prepare_hmeq(hmeq)
print(f"Training: {len(X_train):,} applicants, {y_train.sum()} defaults ({y_train.mean():.2%})")
print(f"Test:     {len(X_test):,} applicants, {y_test.sum()} defaults ({y_test.mean():.2%})")
print(f"Predictors after indicator coding: {X_train.shape[1]}")

### Step 3: Read the output in plain English

In [ ]:
print(f"We trained on {len(X_train):,} applicants and kept {len(X_test):,} sealed away for the final check.")
print(f"Both parts have about the same default rate ({y_train.mean():.1%} and {y_test.mean():.1%}), which is what stratify=y is for.")
print(f"Blanks were filled with training medians, for example DEBTINC {medians['DEBTINC']:.2f} and CLAGE {medians['CLAGE']:.2f}.")
print(f"The model sees {X_train.shape[1]} predictors, including the flag for a missing debt-to-income figure.")

### Step 4: Check against JMP

We write the prepared table to a file with a `Split` column (Train or Test) and a numeric `Validation` column (0 for train, 1 for test). In JMP you will use the `Validation` column as the validation column, so JMP trains and tests on exactly the rows used here. The workbook *Trees and Forests in JMP* gives the steps.

In [ ]:
os.makedirs("exports", exist_ok=True)
prepared = pd.concat([X_train.assign(BAD=y_train, Split="Train", Validation=0),
                      X_test.assign(BAD=y_test, Split="Test", Validation=1)])
prepared.to_csv("exports/hmeq_prepared_for_jmp.csv", index=False)
print(f"Written exports/hmeq_prepared_for_jmp.csv with {len(prepared):,} rows. Do not commit this file.")
print(prepared["Split"].value_counts().to_string())

### Step 5: Say it to a manager

*In one sentence:* why did we keep a flag for a missing debt-to-income figure instead of deleting those applicants?

*Your sentence here*

---
## Part 2: The baseline, logistic regression

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Reuse the prepared training and test tables. |
| 2 | **Run the model** | Fit the logistic regression, score train and test. |
| 3 | **Read the output in plain English** | Read AUC, accuracy and the confusion matrix. Meet the accuracy trap. |
| 4 | **Check against JMP** | Compare the fit with Fit Model, Nominal Logistic in JMP. |
| 5 | **Say it to a manager** | One sentence on what the baseline achieves. |

> **Concept: scaling.** Logistic regression finds its answer step by step, and it copes better when every input is on a similar scale (a loan in thousands and a count of delinquencies are very different sizes). `StandardScaler` rescales each input to average 0 and spread 1. It lives **inside the pipeline**, so it learns its averages from the training rows only. Trees split on thresholds, not sizes, so they need no scaling.

> **Concept: the confusion matrix.** Four boxes: *true negatives* (repays, predicted repays), *false positives* (repays, flagged as a default: a good applicant turned away), *false negatives* (defaults, predicted repays: a loss we approved) and *true positives* (defaults we caught). The two kinds of mistake cost different amounts, and which matters more is a business question.

> **Concept: AUC.** Rank all test applicants from most to least risky. AUC is the chance that a randomly chosen defaulter sits above a randomly chosen repayer. 0.5 is a coin toss, 1.0 is perfect, and a value well below 0.5 means the labels have been put the wrong way round.

### Step 2: Run the model

**What to look for first:** the train AUC and the test AUC side by side. A big gap means the model has memorised the training rows.

In [ ]:
logit = make_models()["Logistic regression"]
logit.fit(X_train, y_train)

p_train = logit.predict_proba(X_train)[:, 1]      # probability of default (column 1)
p_test = logit.predict_proba(X_test)[:, 1]
auc_train = roc_auc_score(y_train, p_train)
auc_test = roc_auc_score(y_test, p_test)
print(f"Logistic regression: train AUC {auc_train:.3f}, test AUC {auc_test:.3f}")

In [ ]:
# The confusion matrix at the usual 0.5 cut-off
tn, fp, fn, tp = confusion_matrix(y_test, p_test >= 0.5).ravel()
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["Actually repaid", "Actually defaulted"],
                   columns=["Predicted repay", "Predicted default"]))
accuracy = (tn + tp) / len(y_test)
always_repay = 1 - y_test.mean()
print(f"\nAccuracy at 0.5: {accuracy:.1%}.   Accuracy of 'approve everyone': {always_repay:.1%}")

### Step 3: Read the output in plain English

In [ ]:
print(f"The model ranks a defaulter above a repayer about {auc_test:.0%} of the time on applicants it has never seen.")
print(f"Train and test AUC are {auc_train:.3f} and {auc_test:.3f}: a gap of {abs(auc_train - auc_test):.3f}, so this simple model is not memorising.")
print(f"At 0.5 it catches {tp} of {tp + fn} defaults ({tp/(tp+fn):.0%}) and wrongly flags {fp} good applicants.")
print(f"Accuracy is {accuracy:.1%}, but approving everyone would score {always_repay:.1%}. Accuracy alone flatters the baseline.")

### Step 4: Check against JMP

In JMP, fit a Nominal Logistic model for `BAD` (Target Level 1) with the same predictors and the `Validation` column as validation. The test AUC should match the value above to about two decimal places; small differences come from the penalty and from how JMP codes categories.

### Step 5: Say it to a manager

*One sentence:* what does the baseline achieve, and why is 88% accuracy not impressive on its own?

*Your sentence here*

---
## Part 3: A decision tree

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Grow a tree on the same training rows. |
| 2 | **Run the model** | Fit a free tree and a limited tree; watch train against test as the tree grows. |
| 3 | **Read the output in plain English** | Read a tree node. Read the top of the tree as a story. |
| 4 | **Check against JMP** | Compare with Partition in JMP: the first question and the AUC. |
| 5 | **Say it to a manager** | Tell the story of the first two questions. |

### How a tree thinks

A tree asks a series of yes/no questions about an applicant, such as *is the debt-to-income figure missing?* Each question splits the applicants into two groups that are as different as possible in their default rates. It keeps asking until it is told to stop. The group an applicant ends up in (a **leaf**) predicts the share of defaults among the training applicants who landed there.

> **How to read a tree node.** Each box in the picture below has four lines.
> - **The question** at the top (for example `DELINQ <= 0.5`). **Yes goes left, No goes right.** Leaf boxes have no question.
> - **gini**: how mixed the group is. 0 means everyone in the box has the same outcome; higher means more mixed.
> - **samples**: how many training applicants are in the box.
> - **value**: the count of `[repaid, defaulted]` in the box. The share of defaults is the second number divided by the total.
> The box colour deepens as the share of defaults rises.

### Step 2: Run the model

**What to look for first:** the size of the free tree (depth and leaves), and its training AUC.

In [ ]:
models = make_models()
tree_free = models["Tree, unconstrained"].fit(X_train, y_train)
tree5 = models["Tree, depth 5"].fit(X_train, y_train)

for name, t in (("Unconstrained tree", tree_free), ("Depth-5 tree", tree5)):
    print(f"{name}: depth {t.get_depth()}, {t.get_n_leaves()} leaves, "
          f"train AUC {roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]):.3f}, "
          f"test AUC {roc_auc_score(y_test, t.predict_proba(X_test)[:, 1]):.3f}")

In [ ]:
# The top of the depth-5 tree, drawn. Only the first three levels are shown so that it is readable.
fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(tree5, max_depth=2, feature_names=list(X_train.columns), class_names=["repaid", "defaulted"],
          filled=True, rounded=True, fontsize=10, ax=ax)
ax.set_title("Depth-5 tree: first three levels")
plt.show()

In [ ]:
# The same top of the tree as text, which is easier to read aloud
print(export_text(tree5, feature_names=list(X_train.columns), max_depth=2))

### Step 3: Read the output in plain English

The cell below turns the first questions of the tree into sentences, straight from the fitted tree. `DEBTINC_missing` is 1 where the figure is blank, so "at most 0.5" means "the figure is present".

In [ ]:
tr = tree5.tree_
names = list(X_train.columns)

def describe(node):
    # Returns the question at a node and the default rate and size of a node
    counts = tr.value[node][0]
    return names[tr.feature[node]], tr.threshold[node], counts[1] / counts.sum(), tr.n_node_samples[node]

f0, t0, r0, n0 = describe(0)
print(f"Question 1 (all {n0:,} applicants, {r0:.1%} default): is {f0} at most {t0:.2f}?")
for answer, child in (("Yes", tr.children_left[0]), ("No", tr.children_right[0])):
    counts = tr.value[child][0]
    print(f"\n  {answer}: {tr.n_node_samples[child]:,} applicants, {counts[1] / counts.sum():.1%} default.")
    if tr.children_left[child] != -1:
        f1, t1, _, _ = describe(child)
        print(f"  Then ask: is {f1} at most {t1:.2f}?")
        for a2, c2 in (("Yes", tr.children_left[child]), ("No", tr.children_right[child])):
            k = tr.value[c2][0]
            print(f"      {a2}: {tr.n_node_samples[c2]:,} applicants, {k[1] / k.sum():.1%} default.")

### The overfitting curve

Now let the tree grow deeper and watch what happens. We score the tree on the training rows and on the test rows at each depth. **This chart is for looking at, not for choosing a depth**: if we picked the depth that did best on the test rows, the test rows would no longer be a fair test. Part 4 shows the honest way to choose.

**What to look for:** the training line keeps climbing. Does the test line keep up? The test line is jagged, so do not read much into a single point.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
curve = []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=1).fit(X_train, y_train)
    curve.append({"depth": d,
                  "train AUC": roc_auc_score(y_train, t.predict_proba(X_train)[:, 1]),
                  "test AUC": roc_auc_score(y_test, t.predict_proba(X_test)[:, 1])})
curve = pd.DataFrame(curve).set_index("depth")
display(curve.round(3))

ax = curve.plot(marker="o", figsize=(8, 4), title="Deeper trees fit the training rows better and the test rows worse")
ax.set_ylabel("AUC")
plt.show()

In [ ]:
best_depth = curve["test AUC"].idxmax()
print(f"The training AUC rises from {curve['train AUC'].iloc[0]:.3f} at depth 1 to {curve['train AUC'].iloc[-1]:.3f} at depth {depths[-1]}.")
print(f"The test AUC peaks at depth {best_depth} ({curve['test AUC'].max():.3f}) and is {curve['test AUC'].iloc[-1]:.3f} at depth {depths[-1]}.")
print(f"These trees have no leaf limit, so the depth-5 test AUC here ({curve.loc[5, 'test AUC']:.3f}) is not the {roc_auc_score(y_test, tree5.predict_proba(X_test)[:, 1]):.3f} of the depth-5 tree above, which must keep at least 50 applicants in every leaf. Neither is a mistake: they are two different trees.")
print("A tree that keeps asking questions memorises the training applicants. That is overfitting.")

### Step 4: Check against JMP

JMP's Partition platform grows trees with its own stopping rules, so do not expect the identical tree. **Compare the first question and the test AUC**, not every split. The workbook gives the steps.

### Step 5: Say it to a manager

*Two sentences:* tell the story of the tree's first question. Then say what overfitting means for a lender.

*Your two sentences here*

### Your turn

Change `DEPTH` below and re-run it for depths 2, 4 and 6. The training AUC will keep rising with depth, so it cannot tell you which depth is best. Look instead for the depth where the **cross-validated AUC** stops improving, and ask whether you could talk a manager through a tree that size. Part 4 shows the proper method.

In [ ]:
DEPTH = 3                       # <- change me (try 2, 4, 6)
small = DecisionTreeClassifier(max_depth=DEPTH, min_samples_leaf=50, random_state=1).fit(X_train, y_train)
cv_auc = cross_val_score(small, X_train, y_train, cv=CV, scoring="roc_auc").mean()      # training rows only
print(f"Depth {DEPTH}: {small.get_n_leaves()} leaves, train AUC {roc_auc_score(y_train, small.predict_proba(X_train)[:, 1]):.3f}, cross-validated AUC {cv_auc:.3f}")

### Talk it through

- A non-technical lending manager could (just about) follow this tree. Is that an advantage? When would being able to explain a decision matter more than a few points of AUC?
- The tree asks about missing data first. What would you do in the business to make sure that field is always filled in?

---
## Part 4: Testing honestly with cross-validation

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use only the training rows. The test rows stay sealed. |
| 2 | **Run the model** | Split the training rows into five folds; each fold takes a turn as the mini test. |
| 3 | **Read the output in plain English** | Read the average and the spread across folds. |
| 4 | **Check against JMP** | In JMP use a validation column; JMP can also do k-fold in some platforms. |
| 5 | **Say it to a manager** | Say why one split is one roll of the dice. |

### Why we need it

One train/test split gives one number, and a different split would give a different number. We also want to **choose settings** (how deep should the tree be?) without peeking at the test rows. **Cross-validation** solves both.

> **Concept: five-fold cross-validation.** Cut the *training* rows into five equal folds. Train on four folds and score on the fifth. Repeat so every fold has been the mini test once. You get five scores. Their average is a steadier estimate than any one split, and their **spread** (the standard deviation) tells you how much the result depends on luck. The real test set is untouched, so it stays an honest final check.

### Step 2: Run it

**What to look for:** the average score and the spread. A model that is steady scores about the same on every fold.

In [ ]:
folds = pd.DataFrame({"fold": range(1, 6),
                      "rows to learn from": [len(X_train) - len(v) for _, v in CV.split(X_train, y_train)],
                      "rows to score": [len(v) for _, v in CV.split(X_train, y_train)]})
folds

In [ ]:
# Choose the tree's depth using the training rows only
depth_table = []
for d in [2, 3, 4, 5, 6, 8, 10]:
    t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=50, random_state=1)
    scores = cross_val_score(t, X_train, y_train, cv=CV, scoring="roc_auc")
    depth_table.append({"depth": d, "CV AUC": scores.mean(), "spread (sd)": scores.std()})
depth_table = pd.DataFrame(depth_table).set_index("depth")
display(depth_table.round(3))

### Step 3: Read the output in plain English

In [ ]:
top = depth_table["CV AUC"].max()
enough = depth_table.index[(top - depth_table["CV AUC"]) <= 0.005][0]
print(f"Cross-validated AUC climbs from {depth_table['CV AUC'].iloc[0]:.3f} at depth 2 to {top:.3f}, then flattens.")
print(f"The first depth within 0.005 of the best is {enough}. Going deeper buys almost nothing and costs readability.")
print(f"Depth 5 is the module's fixed setting, and it is nearly as good ({depth_table.loc[5, 'CV AUC']:.3f} against {depth_table.loc[enough, 'CV AUC']:.3f} at depth {enough}), so the module uses the fixed depth 5 and every model is compared on the same terms.")
print("We chose that depth without ever looking at the test rows. The test set is still sealed.")

### Step 4: Check against JMP

JMP's Partition report shows validation statistics for the `Validation` column you supply. It is a single split, not five folds, so use it to check the *test AUC*, and use cross-validation in Python to check the *stability*.

### Step 5: Say it to a manager

*One sentence:* why is the cross-validated score a better guide than a single test score when choosing a setting?

*Your sentence here*

---
## Part 5: A random forest

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Same training rows, same test rows. |
| 2 | **Run the model** | Grow 300 trees, each on a random sample of the applicants and a random choice of questions. |
| 3 | **Read the output in plain English** | Read the accuracy gain and the two importance charts. Spot where they disagree. |
| 4 | **Check against JMP** | Compare Bootstrap Forest and Column Contributions in JMP. |
| 5 | **Say it to a manager** | Say what you gain and what you give up. |

> **Concept: bagging and the random forest.** One opinionated expert can be wrong. Many slightly different experts, averaged, are steadier. A random forest grows many trees. Each tree learns from a random sample of the training applicants (taken with replacement, called a *bootstrap*; averaging many of them is **bagging**, short for bootstrap aggregating) and, at each question, may only consider a random handful of the columns. The trees disagree in different places, so their votes average out the noise. The forest's predicted probability is the average of its trees' probabilities.
> Boosting is a relative that grows trees one after another, each fixing the last one's mistakes. We name it here and leave it there.

### Step 2: Run the model

In [ ]:
forest = make_models()["Random forest, 300 trees"].fit(X_train, y_train)
print(f"Forest of {len(forest.estimators_)} trees fitted.")
print(f"Train AUC {roc_auc_score(y_train, forest.predict_proba(X_train)[:, 1]):.3f}, "
      f"test AUC {roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]):.3f}")

**How many trees is enough?** More trees cost more time. **What to look for:** where the line flattens.

In [ ]:
size_rows = []
for k in [1, 5, 25, 100, 300]:
    f = RandomForestClassifier(n_estimators=k, min_samples_leaf=5, random_state=1, n_jobs=-1).fit(X_train, y_train)
    size_rows.append({"trees": k, "test AUC": roc_auc_score(y_test, f.predict_proba(X_test)[:, 1])})
forest_size = pd.DataFrame(size_rows).set_index("trees")
display(forest_size.round(3))
print(f"One tree scores {forest_size['test AUC'].iloc[0]:.3f}; {forest_size.index[2]} trees reach {forest_size['test AUC'].iloc[2]:.3f}; "
      f"{forest_size.index[-1]} trees reach {forest_size['test AUC'].iloc[-1]:.3f}. Most of the gain arrives early.")

### Importance, two ways

> **Concept: feature importance.** *Impurity importance* adds up how much each column cleaned up the groups wherever the forest used it. It is quick, but it flatters continuous columns that offer many places to split. *Permutation importance* shuffles one column in the **test** rows and measures how much the AUC drops. A column the model really relies on hurts when shuffled. It is slower and more honest, and it is the one to trust when the two disagree.

**What to look for:** does the same column come top in both? Which columns move?

In [ ]:
impurity = pd.Series(forest.feature_importances_, index=X_train.columns).sort_values(ascending=False)
perm = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=1, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
impurity.head(8)[::-1].plot.barh(ax=axes[0], color="#1A4C3B", title="Impurity importance (built into the forest)")
permutation.head(8)[::-1].plot.barh(ax=axes[1], color="#E07B39", title="Permutation importance (drop in test AUC)")
plt.tight_layout()
plt.show()

### Step 3: Read the output in plain English

In [ ]:
top_imp, top_perm = impurity.index[0], permutation.index[0]
print(f"Top column by impurity: {top_imp} ({impurity.iloc[0]:.3f}). Top by permutation: {top_perm} (AUC drops by {permutation.iloc[0]:.3f}).")
if top_imp == top_perm:
    print("The two methods agree on the most important column.")
only_imp = [c for c in impurity.index[:6] if c not in permutation.index[:6]]
only_perm = [c for c in permutation.index[:6] if c not in impurity.index[:6]]
print(f"In the impurity top six but not the permutation top six: {only_imp if only_imp else 'none'}.")
print(f"In the permutation top six but not the impurity top six: {only_perm if only_perm else 'none'}.")
print("When the two disagree, trust permutation: impurity flatters columns with many possible split points.")

### Step 4: Check against JMP

JMP's Bootstrap Forest report gives Column Contributions, which is its version of impurity importance. Check that the same column tops the list. The forest in JMP is built differently, so expect a similar ranking, not the same numbers.

### Step 5: Say it to a manager

*Two sentences:* what do you gain by using a forest, and what do you give up?

*Your two sentences here*

---
## Part 6: One script to compare every model

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Write one function that takes any set of models and prints one table. |
| 2 | **Run the model** | Run it on the logistic regression, both trees and the forest. |
| 3 | **Read the output in plain English** | Read the table, the ROC picture and the money. |
| 4 | **Check against JMP** | Reproduce the logistic and forest AUCs in JMP. |
| 5 | **Say it to a manager** | Say which model you would back and why. |

### Step 2: Run the comparison

The function below is the whole comparison in one place. It takes a **dictionary of models**, scores every one on the same split in the same way, and returns a table:

- **CV AUC** and its **sd**: five folds on the training rows only.
- **Train AUC** and **Test AUC**.
- **Accuracy at 0.5.**
- **Money** at the lender's cut-off and at 0.5, using the invented costs. Money is a function we pass in, so the same script works for any business.

In [ ]:
def compare(models, X_train, X_test, y_train, y_test, money, cutoff, label):
    rows, test_probs = [], {}
    for name, model in models.items():
        cv = cross_val_score(model, X_train, y_train, cv=CV, scoring="roc_auc")      # training rows only
        model.fit(X_train, y_train)
        p_tr = model.predict_proba(X_train)[:, 1]
        p_te = model.predict_proba(X_test)[:, 1]
        test_probs[name] = p_te
        rows.append({"Model": name,
                     "CV AUC": cv.mean(), "CV sd": cv.std(),
                     "Train AUC": roc_auc_score(y_train, p_tr),
                     "Test AUC": roc_auc_score(y_test, p_te),
                     "Accuracy at 0.5": accuracy_score(y_test, p_te >= 0.5),
                     f"{label} at {cutoff:.3f}": money(y_test, p_te, cutoff),
                     f"{label} at 0.5": money(y_test, p_te, 0.5)})
    return pd.DataFrame(rows).set_index("Model"), test_probs

def loan_cost(y, p, cutoff):
    flagged = p >= cutoff
    missed = ((~flagged) & (y == 1)).sum()          # defaults we approved
    declined = (flagged & (y == 0)).sum()           # good applicants we turned away
    return float(missed * COST_MISSED_DEFAULT + declined * COST_DECLINED_GOOD)

results, test_probs = compare(make_models(), X_train, X_test, y_train, y_test, loan_cost, LENDER_CUTOFF, "Cost")
display(results.round(3).style.format({"Cost at 0.091": "{:,.0f}", "Cost at 0.5": "{:,.0f}"}))

> **Concept: reading an ROC curve.** The curve shows, for every possible cut-off, the share of real defaulters caught (up the side) against the share of good applicants wrongly flagged (along the bottom). The bottom-left corner is a cut-off so high that nobody is flagged; the top-right is a cut-off so low that everybody is. A perfect model shoots straight up the left edge and then across the top. The dashed diagonal is a coin toss. **The further a curve bows towards the top-left, the better.** AUC is the area under the curve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colours = ["#1A4C3B", "#E07B39", "#999999", "#2E86AB"]
for (name, p), colour in zip(test_probs.items(), colours):
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, color=colour, linewidth=2, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Coin toss")
ax.set_xlabel("Good applicants wrongly flagged")
ax.set_ylabel("Defaulters caught")
ax.set_title("ROC curves on the test applicants")
ax.legend(loc="lower right")
plt.show()

### Step 3: Read the output in plain English

**What to look for:** compare **CV AUC** with **Test AUC** (do they agree?), compare **Train AUC** with **Test AUC** (a big gap means memorising), and compare the money columns (what the model is worth).

In [ ]:
cost_col = f"Cost at {LENDER_CUTOFF:.3f}"
best = results["Test AUC"].idxmax()
base = "Logistic regression"
forest_name = "Random forest, 300 trees"
free = "Tree, unconstrained"
saving = results.loc[base, cost_col] - results.loc[forest_name, cost_col]

print(f"Best test AUC: {best} ({results.loc[best, 'Test AUC']:.3f}). The logistic baseline scores {results.loc[base, 'Test AUC']:.3f}.")
print(f"The unconstrained tree scores {results.loc[free, 'Train AUC']:.3f} on the training rows and {results.loc[free, 'Test AUC']:.3f} on the test rows: that is memorising.")
print(f"The forest is steadier across folds (sd {results.loc[forest_name, 'CV sd']:.3f}) than the unconstrained tree (sd {results.loc[free, 'CV sd']:.3f}).")
print(f"At the lender's cut-off the forest costs {results.loc[forest_name, cost_col]:,.0f} against {results.loc[base, cost_col]:,.0f} for the logistic baseline: "
      f"a saving of {saving:,.0f} on {len(y_test):,} applicants, about {saving / len(y_test):,.0f} per applicant (invented costs).")

### A check that fails quietly

A colleague wrote the cell below to confirm that the unconstrained tree is "unreliable on unseen applicants". It runs without any error. Run it, then ask yourself: does the answer look right? Fix it.

In [ ]:
# A colleague's check of the unconstrained tree on unseen applicants
tree_free = make_models()["Tree, unconstrained"].fit(X_train, y_train)
unseen_auc = roc_auc_score(y_train, tree_free.predict_proba(X_train)[:, 1])
print(f"AUC on unseen applicants: {unseen_auc:.3f}")

### Step 4: Check against JMP

Open `exports/hmeq_prepared_for_jmp.csv` in JMP, set `BAD` to nominal with Target Level 1 and use `Validation` as the validation column. Fit a Nominal Logistic model, a Partition tree and a Bootstrap Forest. Compare each **validation AUC** with the **Test AUC** column above. The numbers below are the ones to compare.

In [ ]:
results[["Train AUC", "Test AUC"]].round(3)

### Ask Copilot

The table has accuracy but not **precision** (of the applicants flagged, how many really default) or **recall** (of the real defaulters, how many were flagged). Ask Copilot to add them.

> *Suggested prompt:* "I have a function `compare` that returns a pandas DataFrame of model scores. Add two columns for precision and recall of the test predictions at a cut-off of 0.5, using scikit-learn's precision_score and recall_score. Keep the rest unchanged."

Paste the code Copilot gives you below, run it, and **check** it: recall for the logistic regression should equal the share of defaulters it caught in the confusion matrix in Part 2.

In [ ]:
# Paste and run Copilot's extended comparison here

### Step 5: Say it to a manager (fixed cell)

Two sentences in business language, using what you saw in the table. Say which model you would back and what the evidence leaves unsure.

*Your two sentences here*

### Talk it through

1. **Which model would you deploy, and why?** Think beyond the metrics: who has to explain a declined application, and who is accountable when the model is wrong?
2. **What could go wrong in production?** How would you notice if the model got worse over time?
3. **How would you explain the forest's decision** to someone who has never heard of machine learning?
4. **Launch, change or stop?** If you were the lender, which of these would you launch, change or stop, and what would it cost to be wrong?

---
# Exercise: does the forest beat the simple model on churn? (telecoms customers)

The retention team has the same four models to choose from. Your job is to run the *same* comparison on a different file and see whether the answer is the same.

## Part 7: Look at the customers and prepare them

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Count customers and churners, look at who leaves. |
| 2 | **Run the model** | Split first, indicator-code the categories, keep MonthlyCharges as it is. |
| 3 | **Read the output in plain English** | Check the shapes and the baseline accuracy. |
| 4 | **Check against JMP** | Export the prepared table with a `Split` column. |
| 5 | **Say it to a manager** | Say who leaves, in one sentence. |

### Step 1: Load and look

**What to look for:** the churn rate, and which groups leave most. The workbook has summary sheets first, so we name the data sheet.

`TotalCharges` is not used as a predictor. It is roughly tenure multiplied by the monthly charge, so it would hide the tenure effect.

In [ ]:
telco = pd.read_excel("../data/Telco-Customer-Churn.xlsx", sheet_name="Telco-Customer-Churn")
print(f"{len(telco):,} customers; churn rate {(telco['Churn'] == 'Yes').mean():.1%}")
telco[["tenure", "MonthlyCharges", "Contract", "InternetService", "PaymentMethod", "Churn"]].head()

In [ ]:
GROUP_COLUMN = "Contract"          # <- Your turn: try "InternetService", "PaymentMethod", "SeniorCitizen"
rates = (telco.assign(churned=(telco["Churn"] == "Yes").astype(int))
              .groupby(GROUP_COLUMN)["churned"].agg(["mean", "size"]).sort_values("mean", ascending=False))
rates.columns = ["churn rate", "customers"]
display(rates.style.format({"churn rate": "{:.1%}", "customers": "{:,}"}))
worst = rates["churn rate"].idxmax(); best_group = rates["churn rate"].idxmin()
print(f"By {GROUP_COLUMN}, '{worst}' customers churn at {rates.loc[worst, 'churn rate']:.1%} and '{best_group}' customers at {rates.loc[best_group, 'churn rate']:.1%}.")

### Step 2: Run the preparation

Same rules as before: split first, learn nothing from the test rows. Trees need no scaling, and they do not need a reference level, so every category keeps its own column. The logistic model standardises inside its pipeline.

In [ ]:
def prepare_telco(data):
    y = (data["Churn"] == "Yes").astype(int)
    X = data[["tenure", "MonthlyCharges", "SeniorCitizen", "Partner", "Dependents", "PaperlessBilling",
              "Contract", "InternetService", "PaymentMethod"]]
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, **SPLIT)    # split FIRST
    cats = ["Partner", "Dependents", "PaperlessBilling", "Contract", "InternetService", "PaymentMethod"]
    X_train = pd.get_dummies(X_train, columns=cats, dtype=int)
    X_test = pd.get_dummies(X_test, columns=cats, dtype=int).reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test, y_train, y_test

Xt_train, Xt_test, yt_train, yt_test = prepare_telco(telco)
print(f"Training: {len(Xt_train):,} customers, {yt_train.sum():,} churners ({yt_train.mean():.2%})")
print(f"Test:     {len(Xt_test):,} customers, {yt_test.sum():,} churners ({yt_test.mean():.2%})")
print(f"Predictors after indicator coding: {Xt_train.shape[1]}")

### Step 3: Read the output in plain English

In [ ]:
print(f"We trained on {len(Xt_train):,} customers and kept {len(Xt_test):,} sealed away.")
print(f"If we predicted that nobody leaves we would be right {1 - yt_test.mean():.1%} of the time, so any model has to beat that to be worth having.")

### Step 4: Check against JMP

In [ ]:
prepared_t = pd.concat([Xt_train.assign(Churn=yt_train, Split="Train", Validation=0), Xt_test.assign(Churn=yt_test, Split="Test", Validation=1)])
prepared_t.to_csv("exports/telco_prepared_for_jmp.csv", index=False)
print(f"Written exports/telco_prepared_for_jmp.csv with {len(prepared_t):,} rows. Do not commit this file.")

### Step 5: Say it to a manager

*One sentence:* which customers are the most likely to leave?

*Your sentence here*

---
## Part 8: Run the same comparison

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Reuse the `compare` function from Part 6 without changing it. |
| 2 | **Run the model** | Pass in the same four models and a profit function instead of a cost function. |
| 3 | **Read the output in plain English** | Read the table: does the forest win? |
| 4 | **Check against JMP** | Reproduce the logistic and forest AUCs in JMP. |
| 5 | **Say it to a manager** | Say whether the extra complexity pays. |

### Step 2: Run the comparison

The retention economics are **invented for teaching**: an offer costs 30 per customer contacted, a saved customer is worth 300, and the offer saves 40% of those who would have left. Contact a customer when probability × 0.4 × 300 is more than 30, which means above 0.25.

In [ ]:
OFFER_COST, SAVED_VALUE, SAVE_RATE = 30, 300, 0.4          # invented
CHURN_CUTOFF = OFFER_COST / (SAVE_RATE * SAVED_VALUE)
print(f"Contact customers above a predicted probability of {CHURN_CUTOFF:.2f}")

def retention_profit(y, p, cutoff):
    contacted = p >= cutoff
    would_leave = (contacted & (y == 1)).sum()
    return float(would_leave * SAVE_RATE * SAVED_VALUE - contacted.sum() * OFFER_COST)

results_t, probs_t = compare(make_models(), Xt_train, Xt_test, yt_train, yt_test,
                             retention_profit, CHURN_CUTOFF, "Profit")
display(results_t.round(3).style.format({"Profit at 0.250": "{:,.0f}", "Profit at 0.5": "{:,.0f}"}))

### Step 3: Read the output in plain English

**What to look for:** is any model clearly ahead? Look at the gap between **Train AUC** and **Test AUC** for the forest.

In [ ]:
base, forest_name, tree_name, free = "Logistic regression", "Random forest, 300 trees", "Tree, depth 5", "Tree, unconstrained"
profit_col = f"Profit at {CHURN_CUTOFF:.3f}"
sensible = [base, tree_name, forest_name]
spread_auc = results_t.loc[sensible, "Test AUC"].max() - results_t.loc[sensible, "Test AUC"].min()
profits = results_t.loc[sensible, profit_col]
spread_profit = (profits.max() - profits.min()) / profits.max()

print(f"Test AUC of the logistic model, depth-5 tree and forest: {results_t.loc[base, 'Test AUC']:.3f}, "
      f"{results_t.loc[tree_name, 'Test AUC']:.3f} and {results_t.loc[forest_name, 'Test AUC']:.3f}: within {spread_auc:.3f} of each other.")
print(f"Their profits at the cut-off differ by only {spread_profit:.1%} between best and worst.")
print(f"The forest scores {results_t.loc[forest_name, 'Train AUC']:.3f} on the training rows and {results_t.loc[forest_name, 'Test AUC']:.3f} on the test rows: it has fitted some noise.")
print(f"The unconstrained tree is the outlier: test AUC {results_t.loc[free, 'Test AUC']:.3f}.")
print("On this file the complex model does not pay. On the loan file it did. It depends on the data.")

In [ ]:
forest_t = make_models()["Random forest, 300 trees"].fit(Xt_train, yt_train)
imp_t = pd.Series(forest_t.feature_importances_, index=Xt_train.columns).sort_values(ascending=False)
print("Forest impurity importance, top five:")
print(imp_t.head(5).round(3).to_string())

### Your turn

The forest's `min_samples_leaf` is a dial for how much it may memorise. Change it and see what happens to the test AUC and the profit. Remember: the choice should really be made by cross-validation, not by the test rows. Here we only look, to feel the dial.

In [ ]:
LEAF = 5                       # <- change me (try 1, 20, 50)
f = RandomForestClassifier(n_estimators=300, min_samples_leaf=LEAF, random_state=1, n_jobs=-1).fit(Xt_train, yt_train)
p = f.predict_proba(Xt_test)[:, 1]
print(f"min_samples_leaf={LEAF}: train AUC {roc_auc_score(yt_train, f.predict_proba(Xt_train)[:, 1]):.3f}, "
      f"test AUC {roc_auc_score(yt_test, p):.3f}, profit {retention_profit(yt_test, p, CHURN_CUTOFF):,.0f}")

---
## Part 9: Does the forest earn its place?

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Use the fitted forest and the comparison table. |
| 2 | **Run the model** | Check the forest's ranking quality, then extend the comparison with Copilot. |
| 3 | **Read the output in plain English** | Read the profit by cut-off. |
| 4 | **Check against JMP** | Reproduce one result in JMP. |
| 5 | **Say it to a manager** | Write the fixed two-sentence summary. |

### A check that fails quietly

A colleague scores the forest's ranking quality with the cell below. It runs. Does the number look right? Compare it with the table above, and fix it.

In [ ]:
# A colleague's AUC for the forest on the test customers
forest_t = make_models()["Random forest, 300 trees"].fit(Xt_train, yt_train)
colleague_auc = roc_auc_score(yt_test, forest_t.predict(Xt_test))
print(f"Forest test AUC: {colleague_auc:.3f}")

### Ask Copilot

The profit depends on the cut-off. Ask Copilot to draw the profit across cut-offs for the logistic model and the forest.

> *Suggested prompt:* "I have predicted probabilities for the test customers in a dictionary `probs_t` (model name to array) and a function `retention_profit(y, p, cutoff)`. Plot profit against cut-off from 0.05 to 0.95 for each model, with a vertical line at 0.25."

**Check it:** the profit at 0.25 on your chart should match the table in Part 8.

In [ ]:
# Paste and run Copilot's profit-by-cut-off chart here

### Step 4: Check against JMP

Open `exports/telco_prepared_for_jmp.csv`, set `Churn` to nominal with Target Level 1, use `Validation` as the validation column, and fit the logistic model, a Partition tree and a Bootstrap Forest. Compare the validation AUCs with the table in Part 8. They should be within a few thousandths of each other, which is the tie.

### Step 5: Say it to a manager (fixed cell)

Two sentences in business language: does the forest beat the simple model on churn, and what would you tell the retention team to do?

*Your two sentences here*

### Talk it through

- The forest won clearly on loans and tied on churn. What is different about the two files that might explain this?
- If the retention team asked for the forest anyway, what would you ask them to show you first?

---
## Stretch

This space is blank on purpose. Use Copilot to build something of your own, and check it against a number you already trust. Ideas: add a gradient boosting model to the comparison (name it, run it, say what changed); show the Telco comparison with the cut-off chosen on the training rows; plot the forest's predicted probabilities for churners and non-churners as two histograms.

In [ ]:
# Stretch: your idea here

---
*BAI709 Data Modelling and Machine Learning, Regent's University London*